# Exploración inicial de datos MiBici

**Equipo:** 41  
**Integrante:** Omar Neftali Lopez Tamayo  
**Curso:** TC5061 - MLOps  
**Actividad:** Proyecto MiBici: propuesta y base reproducible

## Objetivo

Realizar una exploración inicial de los datos de MiBici para identificar su estructura, variables disponibles, tipos de datos, valores faltantes y posibles inconsistencias.

Esta exploración servirá como base para definir posteriormente las variables y transformaciones necesarias para los tres problemas del proyecto:

- P1: Regresión para estimar la cantidad de viajes futuros.
- P2: Clasificación para estimar el riesgo de una estación.
- P3: Clustering para agrupar estaciones con patrones de uso similares.

> Este notebook se utilizará únicamente para análisis exploratorio. La lógica reutilizable del proyecto se implementará posteriormente en los módulos de Python del repositorio.

In [14]:
from pathlib import Path
print(Path.cwd())

c:\Users\neflo\TC5061-Proyecto-Equipo41\notebooks


In [15]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw")

viajes = pd.read_csv(DATA_DIR / "datos_abiertos_2026_09.csv")
estaciones = pd.read_csv(DATA_DIR / "nomenclatura_2026_09.csv")

print("Datos cargados correctamente.")
print(f"Viajes: {viajes.shape}")
print(f"Estaciones: {estaciones.shape}")

Datos cargados correctamente.
Viajes: (490056, 8)
Estaciones: (484, 5)


## 1. Inspección inicial del conjunto de datos

En esta sección se revisa la estructura general de los archivos de viajes y estaciones, incluyendo dimensiones, nombres de columnas, tipos de datos y ejemplos de registros.

In [16]:
viajes.head()
estaciones.head()
print("Columnas del dataset de viajes:")
print(viajes.columns.tolist())

print("\nColumnas del dataset de estaciones:")
print(estaciones.columns.tolist())
viajes.info()
estaciones.info()

Columnas del dataset de viajes:
['Viaje_Id', 'Usuario_Id', 'Genero', 'Año_de_nacimiento', 'Inicio_del_viaje', 'Fin_del_viaje', 'Origen_Id', 'Destino_Id']

Columnas del dataset de estaciones:
['id', 'name', 'latitude', 'longitude', 'dpcapacity']
<class 'pandas.DataFrame'>
RangeIndex: 490056 entries, 0 to 490055
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Viaje_Id           490056 non-null  int64  
 1   Usuario_Id         490056 non-null  int64  
 2   Genero             486761 non-null  str    
 3   Año_de_nacimiento  490047 non-null  float64
 4   Inicio_del_viaje   490056 non-null  str    
 5   Fin_del_viaje      490056 non-null  str    
 6   Origen_Id          490056 non-null  int64  
 7   Destino_Id         490056 non-null  int64  
dtypes: float64(1), int64(4), str(3)
memory usage: 29.9 MB
<class 'pandas.DataFrame'>
RangeIndex: 484 entries, 0 to 483
Data columns (total 5 columns):
 #   Column   

In [17]:
print("Valores faltantes en viajes:")
display(viajes.isna().sum())

print("\nValores faltantes en estaciones:")
display(estaciones.isna().sum())

Valores faltantes en viajes:


Viaje_Id                0
Usuario_Id              0
Genero               3295
Año_de_nacimiento       9
Inicio_del_viaje        0
Fin_del_viaje           0
Origen_Id               0
Destino_Id              0
dtype: int64


Valores faltantes en estaciones:


id            0
name          0
latitude      0
longitude     0
dpcapacity    0
dtype: int64

### Observaciones iniciales

El archivo de septiembre de 2026 contiene 490,056 viajes y 8 variables.  
El archivo de nomenclatura contiene 484 estaciones y 5 variables.

Se identificaron valores faltantes principalmente en las variables `Genero` y `Año_de_nacimiento`. Las variables relacionadas con el inicio, fin, origen y destino de los viajes presentan información completa para este periodo.

Las fechas de inicio y fin fueron cargadas inicialmente como texto, por lo que deberán convertirse posteriormente a formato `datetime` para construir variables temporales como hora, día de la semana y duración del viaje.

## 2. Revisión de calidad de los datos

Se revisan registros duplicados, valores atípicos, consistencia de fechas y posibles anomalías en las variables disponibles. El objetivo es identificar qué reglas de limpieza serán necesarias posteriormente en el proceso de ingesta reproducible.

In [18]:
print(f"Filas duplicadas completas: {viajes.duplicated().sum()}")
print(f"Viaje_Id duplicados: {viajes['Viaje_Id'].duplicated().sum()}")

Filas duplicadas completas: 758
Viaje_Id duplicados: 758


In [19]:
print("Año mínimo:", viajes["Año_de_nacimiento"].min())
print("Año máximo:", viajes["Año_de_nacimiento"].max())

viajes.loc[
    viajes["Año_de_nacimiento"] < 1900,
    ["Viaje_Id", "Año_de_nacimiento"]
]

Año mínimo: 199.0
Año máximo: 2010.0


,Viaje_Id,Año_de_nacimiento
23535,45732742,199.0
50442,45764575,199.0
102418,45823398,199.0
329272,46077925,199.0
421162,46178965,199.0
421653,46179468,199.0
422084,46179916,199.0
438805,46198550,199.0


In [20]:
viajes["Inicio_del_viaje"] = pd.to_datetime(
    viajes["Inicio_del_viaje"],
    errors="coerce"
)

viajes["Fin_del_viaje"] = pd.to_datetime(
    viajes["Fin_del_viaje"],
    errors="coerce"
)

print(viajes[["Inicio_del_viaje", "Fin_del_viaje"]].dtypes)

Inicio_del_viaje    datetime64[us]
Fin_del_viaje       datetime64[us]
dtype: object


In [21]:
viajes["duracion_min"] = (
    viajes["Fin_del_viaje"] - viajes["Inicio_del_viaje"]
).dt.total_seconds() / 60

viajes["duracion_min"].describe()

count    490056.000000
mean          9.830591
std           8.708030
min           0.000000
25%           5.133333
50%           8.400000
75%          12.716667
max        2166.683333
Name: duracion_min, dtype: float64

### Observaciones sobre la duración de los viajes

La duración mediana de los viajes es de aproximadamente 8.4 minutos, mientras que el 50 % central de los registros se encuentra aproximadamente entre 5.1 y 12.7 minutos.

Se identificaron posibles valores atípicos. Existen registros con duración igual a cero y al menos un viaje presenta una duración superior a 2,000 minutos. Estos casos deberán analizarse antes de definir las reglas de limpieza del proceso de ingesta.

En esta etapa exploratoria los registros no se eliminan; únicamente se documentan las anomalías para establecer posteriormente criterios reproducibles de tratamiento.

In [22]:
print("Duración <= 0 minutos:",
      (viajes["duracion_min"] <= 0).sum())

print("Duración > 60 minutos:",
      (viajes["duracion_min"] > 60).sum())

print("Duración > 120 minutos:",
      (viajes["duracion_min"] > 120).sum())

print("Duración > 1440 minutos:",
      (viajes["duracion_min"] > 1440).sum())

Duración <= 0 minutos: 171
Duración > 60 minutos: 454
Duración > 120 minutos: 95
Duración > 1440 minutos: 1


In [23]:
viajes.nlargest(
    10,
    "duracion_min"
)[
    ["Viaje_Id", "Inicio_del_viaje", "Fin_del_viaje",
     "Origen_Id", "Destino_Id", "duracion_min"]
]

,Viaje_Id,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,duracion_min
288176,46032585,2026-09-20 11:34:40,2026-09-21 23:41:21,35,57,2166.683333
438370,46198074,2026-09-28 16:22:54,2026-09-29 14:08:43,319,312,1305.816667
218327,45954796,2026-09-15 23:49:13,2026-09-16 18:21:26,49,57,1112.216667
68761,45785118,2026-09-05 21:59:43,2026-09-06 11:12:37,150,369,792.900000
444089,46204283,2026-09-28 19:44:54,2026-09-29 08:56:24,233,234,791.500000
314108,46060770,2026-09-21 19:34:15,2026-09-22 06:59:23,11,18,685.133333
250342,45991004,2026-09-18 07:29:12,2026-09-18 17:28:53,340,296,599.683333
11106,45718252,2026-09-01 18:01:53,2026-09-02 03:46:29,228,228,584.600000
322360,46070464,2026-09-22 09:49:16,2026-09-22 19:22:28,96,119,573.200000
107232,45828852,2026-09-08 22:30:07,2026-09-09 07:28:00,66,66,537.883333


### Hallazgos de duración

Se identificaron 171 registros con duración menor o igual a cero minutos, así como 454 viajes con duración superior a una hora. De estos, 95 superan las dos horas y un registro supera las 24 horas.

Los viajes de mayor duración incluyen casos cuyo fin se registra varias horas después del inicio e incluso durante el día siguiente. En esta etapa no se eliminan estos registros, ya que primero se requiere definir y justificar una regla de tratamiento reproducible.

Estos hallazgos serán considerados posteriormente durante el desarrollo del proceso de limpieza en `dataset.py`.

## 3. Consistencia de estaciones

Se valida que los identificadores utilizados como origen y destino en los viajes correspondan con estaciones registradas en el archivo oficial de nomenclatura.

In [24]:
ids_estaciones = set(estaciones["id"])

origenes_invalidos = ~viajes["Origen_Id"].isin(ids_estaciones)
destinos_invalidos = ~viajes["Destino_Id"].isin(ids_estaciones)

print("Orígenes que no aparecen en nomenclatura:",
      origenes_invalidos.sum())

print("Destinos que no aparecen en nomenclatura:",
      destinos_invalidos.sum())

Orígenes que no aparecen en nomenclatura: 0
Destinos que no aparecen en nomenclatura: 0


In [25]:
print("Estaciones en nomenclatura:",
      estaciones["id"].nunique())

print("Estaciones usadas como origen:",
      viajes["Origen_Id"].nunique())

print("Estaciones usadas como destino:",
      viajes["Destino_Id"].nunique())

Estaciones en nomenclatura: 484
Estaciones usadas como origen: 468
Estaciones usadas como destino: 468


In [26]:
viajes["Origen_Id"].value_counts().head(10)

Origen_Id
51     10923
35      7640
271     7211
11      6062
50      5997
54      5208
52      5172
260     4380
194     3969
38      3939
Name: count, dtype: int64